# Pipeline completo de CV-Decoder

Esta notebook muestra el flujo mínimo del proyecto:

1. preparar un CV de ejemplo en PDF;
2. extraer el texto y medir parseabilidad;
3. detectar secciones;
4. comparar el CV contra una oferta;
5. obtener el diagnóstico JSON con ATS, match, requisitos y similitud.

Requiere el modelo `models/ner_dataturks_sm/`. Si no existe, entrenarlo antes con:

```bash
uv run python scripts/train_ner.py --sources dataturks --schema fine --output ner_dataturks_sm
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from fpdf import FPDF
from fpdf.enums import XPos, YPos
from src.parsing.pdf_reader import read_pdf
from src.parsing.quality import text_quality, is_parseable
from src.core.diagnostico import analizar, procesar_cv


## 1. Crear un CV de ejemplo

Se genera un PDF simple con texto seleccionable para que el pipeline pueda leerlo. No se usan datos personales reales.

In [ ]:
cv_texto = """Alex Morgan
Email: alex.morgan@example.com | Phone: +1 555 0100
LinkedIn: linkedin.com/in/alexmorgan

Professional Summary
Data analyst with 5 years of experience building dashboards, automating reports
and translating business questions into measurable insights.

Experience
Senior Data Analyst - Northwind Retail (2021 - Present)
- Built Power BI dashboards used by 120 store managers.
- Automated weekly reporting with Python and SQL, reducing manual work by 10 hours per week.
- Designed A/B tests and presented results to product and marketing teams.

Data Analyst - Acme Logistics (2019 - 2021)
- Cleaned and modeled operational data with pandas and SQL.
- Created Tableau dashboards for delivery performance.

Skills
Python, SQL, pandas, Power BI, Tableau, statistics, A/B testing, Excel, communication

Education
BSc in Statistics - State University (2019)
"""

pdf_path = ROOT / "data" / "processed" / "notebook_sample_cv.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)

pdf = FPDF()
pdf.set_margins(18, 18, 18)
pdf.set_auto_page_break(True, margin=18)
pdf.add_page()
pdf.set_font("helvetica", size=10)
for linea in cv_texto.splitlines():
    pdf.multi_cell(0, 5, linea or " ", new_x=XPos.LEFT, new_y=YPos.NEXT)
pdf.output(str(pdf_path))

print(f"PDF de ejemplo: {pdf_path}")


## 2. Extraer texto y medir parseabilidad

In [ ]:
leido = read_pdf(pdf_path)
print("Error de lectura:", leido["error"])
print("Páginas:", len(leido["paginas"]), "| Páginas sin texto:", leido["paginas_sin_texto"])
print("Calidad:", text_quality(leido["texto"]))
print("¿Es parseable?:", is_parseable(leido["texto"]))
print()
print(leido["texto"][:600])


## 3. Detectar secciones del CV

In [ ]:
perfil = procesar_cv(leido["texto"])
print(json.dumps(perfil["secciones"], ensure_ascii=False, indent=2))


## 4. Comparar contra una oferta

In [ ]:
oferta = """We are looking for a Data Analyst to join our commercial analytics team.

Responsibilities
- Build dashboards and automated reports with Power BI and SQL.
- Analyze product and sales data to support business decisions.
- Design experiments and communicate findings to non-technical stakeholders.

Requirements
- 3+ years of experience in data analysis.
- Strong SQL and Python skills.
- Experience with pandas, statistics and A/B testing.
- Excellent communication skills.

Nice to have
- Experience with Tableau.
- Knowledge of retail or logistics data.
"""

resultado = analizar(leido["texto"], oferta, motor_similitud="tfidf")
print(json.dumps(resultado, ensure_ascii=False, indent=2))


## 5. Leer el resultado

- `puntaje_ats` mide la calidad del CV como documento.
- `puntaje_match` mide la adecuación a la oferta.
- `requisitos` muestra qué falta y qué se cumple.
- `similitud` explica el parecido textual y por secciones.

Para reproducir la evaluación de calibración completa:

```bash
uv run python scripts/eval_match.py
```